# VITS Training: Hindi Female TTS (Baseline vs Clustered)

**Run All** - fully automated execution.

| Model | Vocab | Tokens |
|-------|-------|--------|
| Baseline VITS | 62 (57 phonemes + 5 special) | `b aa r ax t` |
| Clustered VITS | 44 (39 clusters + 5 special) | `C3 C0 C29 C0 C1` |

Both use **identical** audio, splits, architecture, optimizer, seed, and hyperparameters.


In [ ]:
# ============================================================
# Cell 1: Environment & GPU check
# ============================================================
import sys, platform, torch, os, random
import numpy as np

# Enable deterministic kernels for a controlled comparison.
# Do not disable cuDNN unless a documented, reproducible runtime error requires it.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Enable only if the pinned environment reproduces a documented T4 error.
USE_CUDNN_WORKAROUND = False
if USE_CUDNN_WORKAROUND:
    torch.backends.cudnn.enabled = False
if USE_CUDNN_WORKAROUND and not hasattr(torch.nn.functional, '_orig_conv1d'):
    torch.nn.functional._orig_conv1d = torch.nn.functional.conv1d
    def safe_conv1d(*args, **kwargs):
        prev = torch.backends.cudnn.enabled
        torch.backends.cudnn.enabled = False
        try:
            return torch.nn.functional._orig_conv1d(*args, **kwargs)
        finally:
            torch.backends.cudnn.enabled = prev
    torch.nn.functional.conv1d = safe_conv1d

print(f'Python : {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'OS     : {platform.platform()}')


In [ ]:
# ============================================================
# Cell 2: Install Coqui TTS (community fork) + pin transformers
# ============================================================
!pip install -q "coqui-tts==0.27.5" "transformers==4.57.3"

import importlib.metadata
print('coqui-tts:', importlib.metadata.version('coqui-tts'))
print('transformers:', importlib.metadata.version('transformers'))


In [ ]:
# ============================================================
# Cell 3: Mount Google Drive, extract data to fast local disk
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT  = '/content/drive/MyDrive/SAMSUNG-TTS-EXPERIMENT'
LOCAL_DATA_DIR = '/content/tts_hindi_female'
MODELS_DIR     = f'{DRIVE_PROJECT}/models/tts_hindi_female'
ZIP_PATH       = f'{DRIVE_PROJECT}/data/tts_hindi_female_data.zip'

import os, zipfile, shutil
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

def extract_zip_safely(zip_path, target_dir):
    print(f'Extracting {zip_path} to {target_dir} ...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.infolist():
            # Normalize Windows backslashes to Linux forward slashes
            clean_name = member.filename.replace('\\', '/').lstrip('/')
            if not clean_name:
                continue
            target_path = os.path.join(target_dir, clean_name)
            if member.is_dir() or clean_name.endswith('/'):
                os.makedirs(target_path, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target_path), exist_ok=True)
                with zip_ref.open(member) as source, open(target_path, 'wb') as target:
                    shutil.copyfileobj(source, target)
    print('Extraction complete.')

drive_data_folder = f'{DRIVE_PROJECT}/data/tts_hindi_female'

# Prefer the unzipped folder: it carries the current split manifests.
# The ZIP is only a portable fallback and must be rebuilt after any split change.
if os.path.exists(drive_data_folder):
    print(f'Using current unzipped Drive folder: {drive_data_folder}')
    print(f'Copying dataset to fast local storage ({LOCAL_DATA_DIR}) ...')
    shutil.copytree(drive_data_folder, LOCAL_DATA_DIR, dirs_exist_ok=True)
    print('Copy complete.')
elif os.path.exists(ZIP_PATH):
    extract_zip_safely(ZIP_PATH, LOCAL_DATA_DIR)
else:
    raise FileNotFoundError(
        f'ZIP not found at {ZIP_PATH} and folder not found at {drive_data_folder}\n'
        'Upload tts_hindi_female_data.zip or tts_hindi_female folder to Drive under SAMSUNG-TTS-EXPERIMENT/data/'
    )

DATA_DIR = LOCAL_DATA_DIR

# Keep the training tokenizer identical to repository inference code.
TOKENIZER_SOURCE = f'{DRIVE_PROJECT}/tts/vits_tokenizer.py'
if not os.path.exists(TOKENIZER_SOURCE):
    raise FileNotFoundError(
        f'Missing {TOKENIZER_SOURCE}. Sync the updated tts/ directory to Drive before training.'
    )
shutil.copy2(TOKENIZER_SOURCE, '/content/vits_tokenizer.py')
sys.path.insert(0, '/content')

# Check if wavs directory is named 'processed' and rename if needed
if not os.path.exists(f'{DATA_DIR}/wavs') and os.path.exists(f'{DATA_DIR}/processed'):
    print('Renaming processed/ -> wavs/ for Coqui TTS compatibility...')
    os.rename(f'{DATA_DIR}/processed', f'{DATA_DIR}/wavs')

# Verify data
print('\n--- Data verification ---')
for split in ['train', 'val', 'test']:
    for kind in ['metadata_baseline.csv', 'metadata_clustered.csv']:
        p = f'{DATA_DIR}/{split}/{kind}'
        ok = os.path.exists(p)
        count = sum(1 for _ in open(p)) if ok else 0
        print(f'  {"OK" if ok else "MISSING":7s} {split}/{kind}  ({count} lines)')

wav_dir = f'{DATA_DIR}/wavs'
if os.path.isdir(wav_dir):
    wav_count = len([f for f in os.listdir(wav_dir) if f.endswith('.wav')])
    print(f'  WAVs: {wav_count}')
else:
    raise FileNotFoundError(f'wavs/ directory missing at {wav_dir}')


In [ ]:
# ============================================================
# Cell 4: Vocabularies, custom formatter, tokenizer patch
# ============================================================
import os
from vits_tokenizer import (
    BASELINE_PHONEMES as SHARED_BASELINE_PHONEMES,
    CLUSTER_TOKENS as SHARED_CLUSTER_TOKENS,
    build_vocab as shared_build_vocab,
    patch_tokenizer as shared_patch_tokenizer,
)

# ---------- Token inventories ----------
SPECIAL_TOKENS = ['<pad>', '<bos>', '<eos>', '<blnk>', '<wb>']

BASELINE_PHONEMES = [
    'a','aa','ae','ax','b','bh','c','ch','d','dh',
    'dx','dxh','dxhq','dxq','ee','ei','f','g','gh','gq',
    'h','hq','i','ii','j','jh','k','kh','khq','kq',
    'l','lx','m','mq','n','ng','nj','nx','o','ou',
    'p','ph','q','r','rq','s','sh','sx','t','th',
    'tx','txh','u','uu','w','y','z'
]

CLUSTER_TOKENS = [f'C{i}' for i in range(39)]

def build_vocab(content_tokens):
    vocab = {}
    for i, t in enumerate(SPECIAL_TOKENS):
        vocab[t] = i
    for i, t in enumerate(content_tokens):
        vocab[t] = len(SPECIAL_TOKENS) + i
    return vocab

baseline_vocab  = build_vocab(BASELINE_PHONEMES)
clustered_vocab = build_vocab(CLUSTER_TOKENS)

print(f'Baseline  vocab: {len(baseline_vocab)} tokens')
print(f'Clustered vocab: {len(clustered_vocab)} tokens')

# ---------- Custom Formatter ----------
# Our CSV: filename|token_sequence  (2 columns, no header)
# Coqui's default ljspeech formatter expects 3 columns -> IndexError
from TTS.tts.datasets import register_formatter

def hindi_formatter(root_path, manifest_file, **kwargs):
    """Parse our 2-column pipe-delimited metadata."""
    txt_file = os.path.join(root_path, manifest_file)
    items = []
    with open(txt_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cols = line.split('|', 1)
            if len(cols) < 2:
                continue
            wav_stem = cols[0].strip()
            text     = cols[1].strip()
            wav_path = os.path.join(root_path, 'wavs', wav_stem + '.wav')
            if not os.path.exists(wav_path):
                continue
            items.append({
                'text': text,
                'audio_file': wav_path,
                'speaker_name': 'hindi_female',
                'root_path': root_path,
            })
    return items

try:
    register_formatter('hindi_tts', hindi_formatter)
    print('Custom formatter registered: hindi_tts')
except ValueError:
    from TTS.tts.datasets import formatters
    formatters._FORMATTER_REGISTRY['hindi_tts'] = hindi_formatter
    print('Custom formatter updated: hindi_tts')

# ---------- Tokenizer Monkey-Patch ----------
# Whitespace-based tokenization ensures atomic multi-char tokens like C10 stay intact
def patch_tokenizer(tokenizer, vocab):
    """Replace text_to_ids/ids_to_text with whitespace-aware versions."""
    id_to_tok = {v: k for k, v in vocab.items()}
    pad_id  = vocab['<pad>']
    bos_id  = vocab['<bos>']
    eos_id  = vocab['<eos>']
    blnk_id = vocab['<blnk>']

    def text_to_ids(text):
        tokens = text.strip().split()
        ids = []
        for t in tokens:
            if t in vocab:
                ids.append(vocab[t])
        return ids

    def ids_to_text(ids):
        special = {pad_id, bos_id, eos_id, blnk_id}
        return ' '.join(id_to_tok[i] for i in ids if i in id_to_tok and i not in special)

    tokenizer.text_to_ids = text_to_ids
    tokenizer.ids_to_text = ids_to_text
    tokenizer.vocab_size  = len(vocab)
    tokenizer.pad_id      = pad_id
    tokenizer.blank_id    = blnk_id
    tokenizer.bos_id      = bos_id
    tokenizer.eos_id      = eos_id

    if hasattr(tokenizer, 'characters') and tokenizer.characters is not None:
        try:
            chars = tokenizer.characters
            chars._char_to_id = vocab
            chars._id_to_char = id_to_tok
            chars.char_to_id  = lambda c: vocab.get(c, pad_id)
            chars.id_to_char  = lambda i: id_to_tok.get(i, '')
            chars.vocab_size  = len(vocab)
            chars.num_chars   = len(vocab)
            chars.pad_id      = pad_id
            chars.blank_id    = blnk_id
            chars.pad         = '<pad>'
            chars.blank       = '<blnk>'
        except Exception:
            pass

    return tokenizer

def get_latest_checkpoint(output_dir):
    """Auto-resume training if checkpoint exists."""
    if not os.path.exists(output_dir):
        return None
    ckpts = [
        os.path.join(output_dir, f)
        for f in os.listdir(output_dir)
        if f.startswith('checkpoint_') and f.endswith('.pth')
    ]
    if not ckpts:
        return None
    ckpts.sort(key=lambda x: os.path.getmtime(x))
    return ckpts[-1]

if USE_CUDNN_WORKAROUND and not hasattr(torch.nn.functional, '_orig_conv1d'):
    torch.nn.functional._orig_conv1d = torch.nn.functional.conv1d
    def safe_conv1d(*args, **kwargs):
        prev = torch.backends.cudnn.enabled
        torch.backends.cudnn.enabled = False
        try:
            return torch.nn.functional._orig_conv1d(*args, **kwargs)
        finally:
            torch.backends.cudnn.enabled = prev
    torch.nn.functional.conv1d = safe_conv1d

# The shared implementation is also applied by tts/tts_inference.py.
BASELINE_PHONEMES = list(SHARED_BASELINE_PHONEMES)
CLUSTER_TOKENS = list(SHARED_CLUSTER_TOKENS)
build_vocab = shared_build_vocab
baseline_vocab = build_vocab(BASELINE_PHONEMES)
clustered_vocab = build_vocab(CLUSTER_TOKENS)
patch_tokenizer = shared_patch_tokenizer
print('Shared strict tokenizer utilities ready.')


In [ ]:
# ============================================================
# Cell 5: Smoke tests
# ============================================================
class DummyTokenizer: pass

def make_test_tokenizer(vocab, content_tokens):
    tok = DummyTokenizer()
    tok.characters = None
    return patch_tokenizer(tok, vocab)

tok_b = make_test_tokenizer(baseline_vocab, BASELINE_PHONEMES)
tok_c = make_test_tokenizer(clustered_vocab, CLUSTER_TOKENS)

print('=== TEST 1: Baseline tokenization ===')
ids = tok_b.text_to_ids('b aa r ax t')
assert len(ids) == 5, f'Expected 5, got {len(ids)}'
print(f'  "b aa r ax t" -> {ids}  [PASS]')

print('\n=== TEST 2: C10 is atomic ===')
ids = tok_c.text_to_ids('C0 C10 C38')
assert len(ids) == 3, f'Expected 3, got {len(ids)}'
print(f'  "C0 C10 C38" -> {ids}  [PASS]')

print('\n=== TEST 2b: Unknown tokens fail loudly ===')
try:
    tok_c.text_to_ids('C0 NOT_A_TOKEN')
    raise AssertionError('Unknown token was silently accepted')
except ValueError:
    print('  Unknown token rejection  [PASS]')

print('\n=== TEST 3: <wb> boundary ===')
ids = tok_b.text_to_ids('b aa <wb> r ax')
assert len(ids) == 5
print(f'  "b aa <wb> r ax" -> {ids}  [PASS]')

print('\n=== TEST 4: Dataloader consistency ===')
for split in ['train', 'val']:
    bp = f'{DATA_DIR}/{split}/metadata_baseline.csv'
    cp = f'{DATA_DIR}/{split}/metadata_clustered.csv'
    with open(bp) as f:
        b_names = [l.split('|')[0] for l in f if l.strip()]
    with open(cp) as f:
        c_names = [l.split('|')[0] for l in f if l.strip()]
    assert b_names == c_names, f'{split}: WAV names differ!'
    print(f'  {split}: {len(b_names)} files match  [PASS]')

print('\nALL SMOKE TESTS PASSED')


In [ ]:
# ============================================================
# Cell 6: Shared configuration
# ============================================================
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.configs.shared_configs import CharactersConfig, BaseDatasetConfig
from TTS.config.shared_configs import BaseAudioConfig

AUDIO_CFG = BaseAudioConfig(
    sample_rate=22050,
    fft_size=1024,
    win_length=1024,
    hop_length=256,
    num_mels=80,
    mel_fmin=0,
    mel_fmax=None,
    do_trim_silence=False,
)

SEED      = 42
MAX_STEPS = 15000
EPOCHS    = 520  # ~15,000 steps at batch_size=32 on 928 samples

CHAR_CFG_BASELINE = CharactersConfig(
    characters_class='TTS.tts.utils.text.characters.Graphemes',
    pad='<pad>', eos='<eos>', bos='<bos>', blank='<blnk>',
    characters=' '.join(BASELINE_PHONEMES + ['<wb>']),
    punctuations=''
)

CHAR_CFG_CLUSTERED = CharactersConfig(
    characters_class='TTS.tts.utils.text.characters.Graphemes',
    pad='<pad>', eos='<eos>', bos='<bos>', blank='<blnk>',
    characters=' '.join(CLUSTER_TOKENS + ['<wb>']),
    punctuations=''
)

print(f'Audio SR  : {AUDIO_CFG.sample_rate}')
print(f'Target Epochs : {EPOCHS} (~{MAX_STEPS} steps)')
print(f'Seed      : {SEED}')


In [ ]:
# ============================================================
# Cell 7: Train BASELINE VITS (57 phonemes)
# ============================================================
import time, os
from trainer import Trainer, TrainerArgs
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.tts.datasets import load_tts_samples

print('='*60)
print('TRAINING: BASELINE VITS (57 phonemes)')
print('='*60)

baseline_output = f'{MODELS_DIR}/baseline'
os.makedirs(baseline_output, exist_ok=True)

ds_cfg_b = BaseDatasetConfig(
    formatter='hindi_tts',
    dataset_name='hindi_female_baseline',
    path=DATA_DIR,
    meta_file_train='train/metadata_baseline.csv',
    meta_file_val='val/metadata_baseline.csv',
    language='hi'
)

cfg_b = VitsConfig(
    run_name='vits_hindi_female_baseline',
    output_path=baseline_output,
    audio=AUDIO_CFG,
    batch_size=32,
    eval_batch_size=16,
    num_loader_workers=2,
    num_eval_loader_workers=2,
    print_step=50,
    save_step=500,
    save_n_checkpoints=3,
    save_best_after=500,
    run_eval=True,
    lr_gen=0.0002,
    lr_disc=0.0002,
    optimizer='AdamW',
    optimizer_params={'betas': [0.8, 0.99], 'eps': 1e-9, 'weight_decay': 0.01},
    lr_scheduler='ExponentialLR',
    lr_scheduler_params={'gamma': 0.999875},
    cudnn_benchmark=False,
    epochs=EPOCHS,
    datasets=[ds_cfg_b],
    use_phonemes=False,
    phoneme_language=None,
    phonemizer=None,
    text_cleaner=None,
    characters=CHAR_CFG_BASELINE,
    test_sentences=[
        'b aa r ax t <wb> ee k a <wb> m a h a aa n <wb> d a ee sh a <wb> h a ee',
    ],
)

ap_b = AudioProcessor.init_from_config(cfg_b)
tok_b, cfg_b = TTSTokenizer.init_from_config(cfg_b)
tok_b = patch_tokenizer(tok_b, baseline_vocab)

train_b, eval_b = load_tts_samples(
    ds_cfg_b, eval_split=True,
    eval_split_max_size=cfg_b.eval_batch_size,
    eval_split_size=0.05
)
print(f'Train samples: {len(train_b)}, Eval samples: {len(eval_b)}')

# Re-seed immediately before construction so both conditions receive
# controlled initialization and data-loader randomness.
seed_everything(SEED)
model_b = Vits(cfg_b, ap_b, tok_b, speaker_manager=None)

ckpt_b = get_latest_checkpoint(baseline_output)
if ckpt_b:
    print(f'Resuming Baseline training from checkpoint: {ckpt_b}')

t0 = time.time()
trainer_b = Trainer(
    TrainerArgs(restore_path=ckpt_b, skip_train_epoch=False),
    cfg_b,
    output_path=baseline_output,
    model=model_b,
    train_samples=train_b,
    eval_samples=eval_b,
)
trainer_b.fit()
baseline_duration = time.time() - t0
baseline_steps = getattr(trainer_b, 'total_steps', None)
baseline_best_checkpoint = os.path.join(baseline_output, 'best_model.pth')
if not os.path.exists(baseline_best_checkpoint):
    raise FileNotFoundError('best_model.pth was not produced for baseline training')
print(f'\nBaseline training done in {baseline_duration/3600:.2f} hours')


In [ ]:
# ============================================================
# Cell 8: Train CLUSTERED VITS (39 clusters)
# ============================================================
import time, os
from trainer import Trainer, TrainerArgs
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.tts.datasets import load_tts_samples

print('='*60)
print('TRAINING: CLUSTERED VITS (39 clusters)')
print('='*60)

clustered_output = f'{MODELS_DIR}/clustered'
os.makedirs(clustered_output, exist_ok=True)

ds_cfg_c = BaseDatasetConfig(
    formatter='hindi_tts',
    dataset_name='hindi_female_clustered',
    path=DATA_DIR,
    meta_file_train='train/metadata_clustered.csv',
    meta_file_val='val/metadata_clustered.csv',
    language='hi'
)

cfg_c = VitsConfig(
    run_name='vits_hindi_female_clustered',
    output_path=clustered_output,
    audio=AUDIO_CFG,
    batch_size=32,
    eval_batch_size=16,
    num_loader_workers=2,
    num_eval_loader_workers=2,
    print_step=50,
    save_step=500,
    save_n_checkpoints=3,
    save_best_after=500,
    run_eval=True,
    lr_gen=0.0002,
    lr_disc=0.0002,
    optimizer='AdamW',
    optimizer_params={'betas': [0.8, 0.99], 'eps': 1e-9, 'weight_decay': 0.01},
    lr_scheduler='ExponentialLR',
    lr_scheduler_params={'gamma': 0.999875},
    cudnn_benchmark=False,
    epochs=EPOCHS,
    datasets=[ds_cfg_c],
    use_phonemes=False,
    phoneme_language=None,
    phonemizer=None,
    text_cleaner=None,
    characters=CHAR_CFG_CLUSTERED,
    test_sentences=[
        'C3 C0 C29 C0 C1 <wb> C11 C19 C0 <wb> C22 C0 C15 C0 C1 C24 <wb> C7 C0 C11 C31 C0 <wb> C15 C0 C11',
    ],
)

ap_c = AudioProcessor.init_from_config(cfg_c)
tok_c, cfg_c = TTSTokenizer.init_from_config(cfg_c)
tok_c = patch_tokenizer(tok_c, clustered_vocab)

train_c, eval_c = load_tts_samples(
    ds_cfg_c, eval_split=True,
    eval_split_max_size=cfg_c.eval_batch_size,
    eval_split_size=0.05
)
print(f'Train samples: {len(train_c)}, Eval samples: {len(eval_c)}')

seed_everything(SEED)
model_c = Vits(cfg_c, ap_c, tok_c, speaker_manager=None)

ckpt_c = get_latest_checkpoint(clustered_output)
if ckpt_c:
    print(f'Resuming Clustered training from checkpoint: {ckpt_c}')

t0 = time.time()
trainer_c = Trainer(
    TrainerArgs(restore_path=ckpt_c, skip_train_epoch=False),
    cfg_c,
    output_path=clustered_output,
    model=model_c,
    train_samples=train_c,
    eval_samples=eval_c,
)
trainer_c.fit()
clustered_duration = time.time() - t0
clustered_steps = getattr(trainer_c, 'total_steps', None)
clustered_best_checkpoint = os.path.join(clustered_output, 'best_model.pth')
if not os.path.exists(clustered_best_checkpoint):
    raise FileNotFoundError('best_model.pth was not produced for clustered training')
print(f'\nClustered training done in {clustered_duration/3600:.2f} hours')


In [ ]:
# ============================================================
# Cell 9: Save training log
# ============================================================
import json, sys

log = {
    'gpu_type': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'python_version': sys.version,
    'pytorch_version': torch.__version__,
    'coqui_tts_version': importlib.metadata.version('coqui-tts'),
    'transformers_version': importlib.metadata.version('transformers'),
    'random_seed': SEED,
    'max_steps': MAX_STEPS,
    'batch_size': 32,
    'lr_gen': 0.0002,
    'lr_disc': 0.0002,
    'audio_config': {
        'sample_rate': 22050, 'fft_size': 1024, 'win_length': 1024,
        'hop_length': 256, 'num_mels': 80,
    },
    'baseline': {
        'vocab_size': len(baseline_vocab),
        'training_hours': baseline_duration / 3600 if 'baseline_duration' in locals() else None,
        'completed_steps': baseline_steps if 'baseline_steps' in locals() else None,
        'best_checkpoint': baseline_best_checkpoint if 'baseline_best_checkpoint' in locals() else None,
    },
    'clustered': {
        'vocab_size': len(clustered_vocab),
        'training_hours': clustered_duration / 3600 if 'clustered_duration' in locals() else None,
        'completed_steps': clustered_steps if 'clustered_steps' in locals() else None,
        'best_checkpoint': clustered_best_checkpoint if 'clustered_best_checkpoint' in locals() else None,
    },
}

log_path = f'{MODELS_DIR}/training_log.json'
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2, default=str)

print(json.dumps(log, indent=2, default=str))
print(f'\nLog saved: {log_path}')
print('\n*** TRAINING COMPLETE ***')
